# Database preparation

Create sql connection

In [25]:
import sqlite3

# connect to database (this creates gym.db if it does not exist)
conn = sqlite3.connect("gym.db")

# cursor to execute SQL
cur = conn.cursor()

Execute schema and data to make the database ready for use

In [26]:
# Execute schema.sql
with open("schema.sql", "r", encoding="utf-8") as f:  # Reads the file
    schema_sql = f.read()

cur.executescript(schema_sql)  # runs sql statements
conn.commit()  # saves changes

In [27]:
# Execute data.sql
with open("data.sql", "r", encoding="utf-8") as f:
    data_sql = f.read()

cur.executescript(data_sql)
conn.commit()

Now we can query from the database

In [28]:
# Query the database
cur.execute("SELECT * FROM users")

rows = cur.fetchall()
for row in rows:
    print(row)

(1, 'Johnny', 'Nordmann', 'johnny@stud.ntnu.no', '40100001', 1)
(2, 'Viola', 'Smith', 'viola@stud.ntnu.no', '40100010', 1)
(3, 'Erik', 'Hansen', 'erik.hansen@stud.ntnu.no', '40100002', 1)
(4, 'Sara', 'Berg', 'sara.berg@stud.ntnu.no', '40100003', 1)
(5, 'Lucas', 'Johansen', 'lucas.johansen@stud.ntnu.no', '40100004', 0)
(6, 'Emma', 'Larsen', 'emma.larsen@stud.ntnu.no', '40100005', 1)
(7, 'Noah', 'Dahl', 'noah.dahl@stud.ntnu.no', '40100006', 0)
(8, 'Maja', 'Nilsen', 'maja.nilsen@stud.ntnu.no', '40100007', 1)
(9, 'Oliver', 'Strand', 'oliver.strand@stud.ntnu.no', '40100008', 0)
(10, 'Ingrid', 'Moen', 'ingrid.moen@stud.ntnu.no', '40100009', 1)


# Interface

We can use Python as application layer to receive user input, call SQL queries, and show results, i.e. an interface between user and database.

## Book a session

In [32]:
import pandas as pd
query = """SELECT 
                g.gym_name AS Gym,
                a.name,
                a.description,
                s.sessiondate AS Day,
                s.start_time AS Time
                --s.signup_deadline
            FROM group_sessions s
            INNER JOIN activities a ON a.id = s.activity_id
            INNER JOIN halls h ON h.id = s.hall_id
            INNER JOIN gyms g ON g.id = h.gym_id"""

df = pd.read_sql_query(query, conn)

df

,Gym,name,description,Day,Time
0,Øya,Spin 8x3min,Intervalltime på sykkel der du jobber i interv...,2026-03-16,18:00
1,Dragvoll,Spin45,Spinningtime med variert løype fordelt på 2-3 ...,2026-03-16,16:00
2,Øya,Spin60,Spinningtime med variert løype fordelt på 2-4 ...,2026-03-17,18:30
3,Gloeshaugen,Spin 4x4,"Intervalltime på sykkel med god oppvarming, de...",2026-03-17,17:00
4,Øya,Spin 4x4,"Intervalltime på sykkel med god oppvarming, de...",2026-03-18,19:00
5,Dragvoll,Spin45,Spinningtime med variert løype fordelt på 2-3 ...,2026-03-18,16:00


In [38]:
from datetime import datetime
booking_time = datetime.now()
datetime_object = str((datetime.now()).replace(microsecond=0))
date_part = booking_time.date()
time_part = (booking_time.time()).replace(microsecond=0)
# time_part = (booking_time.time()).replace(second=0, microsecond=0)
print(booking_time)
print(date_part)
print(time_part)
print(datetime_object)
print(type(datetime_object))


2026-03-14 18:06:28.898571
2026-03-14
18:06:28
2026-03-14 18:06:28
<class 'str'>


In [ ]:
#### Book session
from datetime import datetime


def book_session(user_id, session_id):
    '''
    Function that performs a booking
    --------------------
    user_id : Users unique id
    session_id: Sessions unique id (is retrieved from ****)
    '''

    #### First check if available spots by comparing booked spots with capacity
    cur.execute("""SELECT COUNT(*) FROM session_bookings WHERE session_id = ?;""", (session_id))
    booked = cur.fetchone()[0]

    cur.execute("""SELECT capacity FROM halls JOIN group_sessions ON halls.id = group_sessions.hall_id WHERE group_sessions.id = ?;""", (session_id))

    capacity = cur.fetchone()[0]

    if booked >= capacity:
        print("Session is full")
        return

    # If did not return above, means it was available spots so booking is performed:
    booking_time = str((datetime.now()).replace(microsecond=0))  # time

    cur.execute("""INSERT INTO session_booking VALUES (?, ?, ?);""", (user_id, session_id, booking_time))

    conn.commit()

    print("Booking successful")

In [ ]:
#### Register when user doesn't show up

In [ ]:
#### Check if user has three dots and should be blacklisted

In [ ]:
#### 

# Reset database during development for recreation

In [24]:
#### Reset database during development so that we can recreate it from scratch when needed

import os

conn.close()  # Must remove db connection first

if os.path.exists("gym.db"):
    os.remove("gym.db")